# BLDC MOTORS

## DC Motors: Brushed (BDC) vs Brushless (BLDC) -concetto teorico- 

Il principio fondamentale per generare una coppia motrice continua è la **commutazione**: la necessità di invertire ciclicamente la direzione del campo magnetico mentre il rotore gira. 
La differenza tra motori *Brushed* e *Brushless* risiede interamente in *come* avviene questa commutazione: in modo **meccanico** (BDC) o in modo **elettronico** (BLDC).

---

<table style="width: 100%; text-align: left; border-collapse: collapse;">
<tr style="background-color: #f8f9fa; border-bottom: 2px solid #dee2e6;">
<th style="padding: 15px; width: 50%; font-size: 1.2em; color: #0056b3;">Brushed DC Motor (BDC)</th>
<th style="padding: 15px; width: 50%; font-size: 1.2em; color: #28a745;">Brushless DC Motor (BLDC)</th>
</tr>
<tr>
<td style="padding: 20px; text-align: center; border-right: 1px solid #dee2e6;">
<img src="BDC.gif" width="70%" alt="Animazione Brushed Motor">
<br><p><em>Commutazione Meccanica</em></p>
</td>
<td style="padding: 20px; text-align: center;">
<img src="BLDC.gif" width="70%" alt="Animazione Brushless Motor">
<br><p><em>Commutazione Elettronica</em></p>
</td>
</tr>
<tr>
<td style="padding: 15px; vertical-align: top; border-right: 1px solid #dee2e6;">
<strong>Architettura Hardware:</strong>
<ul>
<li><strong>Statore (fisso):</strong> Magneti permanenti.</li>
<li><strong>Rotore (mobile):</strong> Avvolgimenti in rame (bobine).</li>
</ul>
<br>
<strong>Principio di Funzionamento:</strong><br>
La commutazione avviene fisicamente. La corrente passa dalla batteria al rotore in movimento attraverso dei contatti striscianti in carbonio chiamati <strong>Spazzole (Brushes)</strong>, che sfregano contro un anello segmentato chiamato <strong>Commutatore</strong>. Mentre il motore gira, le spazzole toccano segmenti diversi, invertendo la polarità del campo magnetico e mantenendo la rotazione.
<br><br>
<strong>Controllo:</strong><br>
Estremamente semplice. Basta applicare una tensione DC per farlo girare. Per controllare velocità e direzione basta un driver ad H-Bridge (PWM).
<br><br>
<strong>Svantaggi:</strong><br>
Le spazzole generano attrito meccanico (scintille e calore), limitando l'efficienza, la velocità massima e creando usura fisica e rumore elettromagnetico (EMI).
</td>
<td style="padding: 15px; vertical-align: top;">
<strong>Architettura Hardware:</strong>
<ul>
<li><strong>Statore (fisso):</strong> Avvolgimenti in rame (solitamente 3 fasi).</li>
<li><strong>Rotore (mobile):</strong> Magneti permanenti.</li>
</ul>
<br>
<strong>Principio di Funzionamento:</strong><br>
La commutazione avviene elettronicamente. Avendo spostato gli avvolgimenti sullo statore fisso, non servono più contatti striscianti. Il campo magnetico rotante viene generato accendendo e spegnendo ciclicamente le bobine fisse tramite un inverter trifase, che "tira" i magneti del rotore facendolo girare.
<br><br>
<strong>Controllo:</strong><br>
Complesso. Richiede un microcontrollore e sensori di posizione (es. Encoder o sensori di Hall) per conoscere l'angolo esatto del rotore e sincronizzare l'iniezione di corrente nelle 3 fasi (controllo FOC/MIT).
<br><br>
<strong>Vantaggi:</strong><br>
Assenza di usura meccanica, alta densità di coppia, altissima efficienza elettrica, ottima dissipazione termica (le bobine sono all'esterno) e trasparenza meccanica.
</td>
</tr>
</table>

## PID (divagazione teorica)
[COSA e COME](https://youtube.com/shorts/U_bn_37bDHA?si=MM8qcAzYO9fDKlgy)<br>
[PID (Graphic construction)](https://youtube.com/shorts/lN5ugarXSkM?si=_4HOPww1hi78XcvN)<br>
[I trust my indian bro](https://youtu.be/000S4xF7EJQ?si=PGiCFFab_xF5Nv0S)

### Approfondimento Teorico: Controllori PID e PD a confronto

Per capire a fondo la differenza tra Servo Mode e MIT Mode, è utile richiamare la teoria del controllo in retroazione. L'obiettivo del controllore è valutare continuamente l'errore $e(t)$ — la differenza tra il setpoint desiderato e la grandezza reale misurata dai sensori — per elaborare un segnale di comando $u(t)$ capace di annullarlo.

#### 1. Il Controllore PID (Proporzionale - Integrale - Derivativo)

Nel Servo Control, l'obiettivo primario è l'inseguimento rigido della traiettoria (*set-point tracking*) e la reiezione dei disturbi. Per farlo, si utilizza la formulazione completa del PID:

$$ u(t) = K_p e(t) + K_i \int_{0}^{t} e(\tau) d\tau + K_d \frac{de(t)}{dt} $$

<div align="center">
    <br>
    <em>
    <div align="center">
    <img src="PID.png" width="90%" alt="Pannello di Telemetria e Grafici CubeMars">
    <p><em>Schema a blocchi classico di un PID.</em></p>
</div>
    </em>
    <br><br>
</div>

Il segnale di controllo è la somma di tre contributi distinti:
*   **Azione Proporzionale (Presente):** Agisce in base all'errore istantaneo. Funziona meccanicamente come una molla: maggiore è la distanza dal target, maggiore è la forza di richiamo. Un $K_p$ troppo elevato rende il sistema reattivo, ma ne compromette la stabilità innescando oscillazioni.
*   **Azione Integrale (Passato):** Tiene traccia della storia dell'errore. Se a causa degli attriti statici il motore si ferma in prossimità del bersaglio senza raggiungerlo, l'azione proporzionale da sola non avrebbe la forza per smuoverlo. L'azione integrale accumula questo piccolo errore nel tempo, aumentando progressivamente la spinta fino ad azzerare completamente l'errore a regime.
*   **Azione Derivativa (Futuro):** Si basa sulla velocità di variazione dell'errore. Si comporta come uno smorzatore viscoso: frena l'attuatore quando questo si sta avvicinando velocemente al setpoint, riducendo o eliminando la sovraelongazione (*overshoot*) dovuta all'inerzia del carico.

#### 2. Il Controllore PD nel MIT Mode (Controllo di Impedenza)

Nell'ambito della robotica indossabile, la sicurezza e la cedevolezza (*compliance*) sono requisiti ben più stringenti della precisione assoluta di posizionamento. Per questo motivo, nel MIT Mode l'azione integrale viene deliberatamente rimossa ($K_i = 0$). 

La legge di controllo si riduce all'emulazione di un sistema massa-molla-smorzatore passivo, a cui si somma un termine di feedforward $\tau_{ff}$:

$$ \tau = K_p (p_{des} - p) + K_d (v_{des} - v) + \tau_{ff} $$

<div align="center">
    <br>
    <em>
    <div align="center">
    <img src="vibrazioni-smorzate-esempio.jpg" width="90%" alt="Pannello di Telemetria e Grafici CubeMars">
    <p><em>Modello fisico massa-molla-smorzatore.</em></p>
</div>
    </em>
    <br><br>
</div>

**Il problema dell'Azione Integrale nella robotica collaborativa**
L'inclusione del termine integrale è estremamente pericolosa quando l'attuatore interagisce fisicamente con l'uomo. Se l'utente oppone resistenza e blocca il movimento dell'esoscheletro, si genera un errore di posizione persistente. Un controllore PI/PID continuerebbe a integrare questo errore, incrementando inesorabilmente la coppia erogata fino alla saturazione termica o meccanica del motore, con l'elevato rischio di ferire l'operatore.

Rimuovendo l'integrale, si accetta matematicamente la presenza di un errore a regime, ma si ottiene in cambio la **trasparenza meccanica**. Il motore perde la sua "rigidità" e si trasforma in un elemento elastico programmabile: 
*   **$K_p$ (Stiffness):** Definisce la rigidezza della molla virtuale.
*   **$K_d$ (Damping):** Fornisce lo smorzamento viscoso che dissipa energia, stabilizzando il movimento e impedendo reazioni brusche.

---

## Interfaccia Operativa CubeMars: Panoramica e Architettura di Controllo

L'interfaccia **CubeMars Tool** è il software di diagnostica, parametrizzazione e controllo per gli attuatori della serie AK. Interfacciamento seriale con il microcontroller integrato nel motore (inviata via USB e tradotta in segnali CAN dal modulo R-LINK).

---


## Cablaggio (Hardware) e introduzione interfaccia

Per avviare la configurazione è necessario un interfacciamento fisico tra la logica di controllo (R-LINK) e il circuito di potenza (batteria e motore).




<div align="center">
    <img src="./cablaggio.png" width="80%" alt="Schema di cablaggio hardware (R-LINK e Cavi di Potenza)">
    <p><em>Fig 1: Cablaggio di potenza e comunicazione seriale verso il motore.</em></p>
</div>

L'interfaccia è logicamente divisa in tre macro-aree funzionali:

<table>
  <tr>
    <td valign="middle" width="80px">
      <img src="./system.png" width="70px">
    </td>
    <td valign="middle">
      <strong>Gestione Sistema (Sinistra):</strong> Interazione con la memoria flash. Permette di salvare, caricare o esportare i parametri di nodo (es. Baud rate, PID gains, CAN ID).
    </td>
  </tr>
  <tr>
    <td valign="middle" width="80px">
      <img src="./Oscilloscope.png" width="70px">
    </td>
    <td valign="middle">
      <strong>Oscilloscopio / Telemetria (Centro):</strong> Visualizza in tempo reale i transitori e le risposte a regime di Corrente, Temperatura, Velocità e Posizione del rotore.
    </td>
  </tr>
  <tr>
    <td valign="middle" width="80px">
      <img src="./zapping.png" width="70px">
    </td>
    <td valign="middle">
      <strong>Telecomando e Manometri (Destra):</strong> Indicatori istantanei (Gauge) e controlli attivi per il pilotaggio (Servo Mode / MIT Mode) e l'arresto di emergenza.
    </td>
  </tr>
</table>

---

## 2. DIGGING IN

Per il pilotaggio di motori BLDC, il firmware a bordo esegue algoritmi di controllo vettoriale ad alta frequenza. 

### A. Gestione Memoria e Stato
*   **Read / Write Parameters:** Eseguono il *pull* o il *push* dei parametri di controllo.
*   **Export / Import Settings:** Clonano la configurazione su un file locale (per garantire simmetria tra anca destra e sinistra).
*   **System Reset:** Forza il riavvio del microcontroller interno.

### B. Area di Telemetria e Analisi Dinamica
Area di *data-logging* per il tuning dei controllori PID e la validazione del sistema:

<div align="center">
    <img src="Interface.png" width="90%" alt="Pannello di Telemetria e Grafici CubeMars">
    <p><em>Fig 2: L'interfaccia centrale mostra i log in tempo reale per la diagnosi del sistema.</em></p>
</div>
<br>

*   **Current:** Monitora la corrente assorbita. La divergenza tra *Current in* e *Current motor* indica perdite per effetto Joule.
*   **RPM & Rotor Position:** Feed diretto dell'encoder (utile per calcolare errore a regime e tempo di assestamento).
*   **FOC (Field Oriented Control):** Visualizza le correnti trasformate (Clarke/Park). L'obiettivo del controllo è ottimizzare la corrente in quadratura $I_q$ (generazione di coppia utile) e azzerare la corrente diretta $I_d$.

### C. SCADA e Loop di Controllo

> ⚠️ **E-STOP:** I bottoni inferiori (punto esclamativo e Stop) fungono da *Kill Switch*. Bloccano immediatamente i gate del ponte Inverter, diseccitando le fasi del motore (*Free-wheel mode*).

L'interfaccia SCADA di CubeMars permette di iniettare comandi in tempo reale per validare la risposta del motore prima di scrivere il firmware sul microcontrollore esterno (Teensy). Implementa due filosofie di controllo diametralmente opposte:

**1. Servo Mode (Nested Loops)**
Si basa su anelli di controllo in cascata (posizione $\rightarrow$ velocità $\rightarrow$ corrente) per un inseguimento rigido del setpoint (*Set-Point Tracking*). Se un ostacolo blocca il motore, il PID saturerà la coppia al massimo consentito: un comportamento potenzialmente pericoloso in un esoscheletro, poiché contrasterebbe rigidamente il movimento umano.

*   **Generatore di Traiettoria:**
    *   `des P`: *Desired Position*. L'angolo di destinazione target.
    *   `des S`: *Desired Speed*. Il limite massimo di velocità di rotazione per il raggiungimento del setpoint.
    *   `des A`: *Desired Acceleration*. La rampa di accelerazione per limitare i picchi di corrente (Jerk).
*   **Controlli Operativi:**
    *   `Multi Mode` / `Single Mode`: Gestione finecorsa software. *Single* confina il rotore tra 0-360°; *Multi* traccia i giri multipli (utile in presenza di un riduttore meccanico).
    *   `Set Origin` & `Go to Origin`: Rispettivamente per eseguire l'homing (salvare lo 0 assoluto) e comandare il ritorno automatico all'origine.
    *   `Slider (▶)`: Cursori laterali per iniettare istantaneamente il setpoint e fare step-testing rapido.

*   **Limiti Hardware e Saturazioni (Safety Constraints):**
    (`T, P, I, B, S, D`) rappresentano i **limiti fisici invalicabili** (saturazioni o *clamps*) imposti agli anelli di controllo in cascata. Nello specifico:
    
    *   `T` **(Torque Limit - $N \cdot m$):** Il limite assoluto di coppia meccanica. Se il carico (es. la gamba dell'utente) oppone una resistenza maggiore, il driver "taglia" l'erogazione impedendo al motore di spingere oltre questo valore di sicurezza.
    *   `P` **(Position Range):** Il limite di escursione spaziale. Definisce un range di confino (software end-stop) oltre il quale il motore non può ruotare, evitando collisioni o iper-estensioni meccaniche del giunto.
    *   `I` **(Input/Phase Current Limit - $A$):** La massima corrente erogabile agli avvolgimenti. Preserva lo statore dal surriscaldamento (effetto Joule) e protegge i MOSFET di potenza da sovraccarichi.
    *   `B` **(Braking/Regen Current - $A$):** Il limite di corrente rigenerativa in frenata. Quando il motore decelera bruscamente agisce da generatore. Questo limite protegge la batteria (o l'alimentatore) da picchi di corrente in ingresso troppo violenti.
    *   `S` **(Speed Limit - $ERPM$):** Il limite di velocità espresso in *Electrical RPM*. L'ERPM si ottiene moltiplicando gli RPM meccanici per il numero di coppie polari del motore. È la saturazione dell'anello di velocità.
    *   `D` **(Duty Cycle Limit - $0.0 \dots 1.0$):** Limita la percentuale massima di tensione della batteria che il ponte Inverter può applicare istantaneamente alle fasi del motore (es. `0.20` = massimo 20% della tensione disponibile). Previene assorbimenti letali durante transitori troppo ripidi.

<br>

**2. MIT Mode (Controllo di Impedenza)**
Sviluppata dal *Biomimetic Robotics Lab del MIT*, questa modalità rivoluziona il paradigma: non cerca la precisione assoluta della posizione, ma regola l'**impedenza** (la relazione dinamica tra motore e ambiente). Utilizza un controllo PD parallelo unito a un Feedforward di coppia per garantire *trasparenza meccanica* (backdrivability). È la modalità operativa utilizzata in OpenExo.

<div align="center">
    <img src="MITpanel.png" width="40%" alt="Pannello dei comandi MIT Mode">
    <p><em>Fig 3: Interfaccia per l'inserimento dei setpoint e dei guadagni PD.</em></p>
</div>

L'equazione governante calcolata a bordo del driver inverter è:

$$\tau = K_p (p_{des} - p) + K_d (v_{des} - v) + \tau_{ff}$$

*   **Variabili di Stato e Impedenza:**
    *   `des P` ($p_{des}$): Posizione di riposo della molla virtuale.
    *   `des S` ($v_{des}$): Velocità target. Spesso tenuta a $0$ nei task stazionari per far sì che lo smorzamento si opponga ai movimenti non voluti.
    *   `des T` ($\tau_{ff}$): Coppia desiderata in Feedforward. Si aggiunge in uscita indipendentemente dall'errore. Ideale per compensare costantemente la gravità.
    *   `KP` (Stiffness): Il "$K$" della molla virtuale. Determina la rigidezza del giunto verso $p_{des}$.
    *   `KD` (Damping): L'attrito viscoso virtuale. Frena i movimenti improvvisi rendendo il giunto più "duro" da muovere velocemente.
*   **Controlli Operativi:**
    *   `ID`: Identificativo del nodo CAN (es. 1) a cui indirizzare i comandi sul bus condiviso.
    *   `RUN` & `EXIT`: `RUN` chiude il loop di controllo e attiva i MOSFET; `EXIT` diseccita le fasi e manda l'inverter in stato di sicurezza (*Safe / Free-wheel*).
    *   `Set Origin`: Resetta il conteggio dell'encoder sull'angolo fisico attuale.
    *   `DEBUG`: Abilita l'invio prolisso di log diagnostici sulla porta seriale per il troubleshooting.
    *   `MOTOR_STATE`: Variabile read-only di diagnostica (un valore `normal` indica l'assenza di errori hardware come *Overcurrent* o perdite di encoder).

> 💡 **Applicazione in OpenExo:** La magia dell'esoscheletro sta nel variare in tempo reale questi parametri in base al ciclo del passo. Durante la spinta, il controller alza $K_p$ ed eroga coppia. Durante la fase di oscillazione (swing) della gamba, il controller abbassa $K_p$ a 0, rendendo il motore completamente "trasparente" e permettendo all'utente di muovere l'arto senza percepire alcuna resistenza meccanica.
